# Cosmos 3 Nano Reasoner ? MMAD Locked-140 on Kaggle T4?2

This is the Phase-1 reportable zero-shot run: **20 questions from each of MMAD?s seven tasks (140 total)** selected by the frozen seed `20260729`.

- Model: community Cosmos 3 Nano Reasoner-only BNB8 checkpoint.
- Backend: Kaggle T4?2 with `device_map="auto"`.
- Decoding: deterministic greedy decoding (`do_sample=False`).
- Outputs: raw reasoning, final response, parsed answer, ground truth, correctness, latency and failure status.
- Safety: labels are never included in the prompt; predictions append immediately to JSONL and resume by `sample_id`.

Run all cells in order. Inspect the five-sample smoke output before running the locked benchmark cell.


In [ ]:
import sys, sysconfig, shutil, subprocess
from pathlib import Path
purelib = Path(sysconfig.get_paths()['purelib'])
shutil.rmtree(purelib / 'PIL', ignore_errors=True)
for stale in list(purelib.glob('Pillow-*.dist-info')) + list(purelib.glob('pillow-*.dist-info')):
    shutil.rmtree(stale, ignore_errors=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--force-reinstall', '--no-deps', 'pillow==11.3.0'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers>=5.14.0', 'accelerate', 'bitsandbytes>=0.49.0', 'qwen-vl-utils', 'safetensors', 'remotezip', 'requests'], check=True)
print('Install complete. NOW restart the Kaggle session once; do not rerun this cell after restart.')

In [ ]:
import os, sys, json, time, shutil, subprocess, random
from pathlib import Path

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['HF_HOME'] = '/kaggle/working/hf_cache'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

WORK = Path('/kaggle/working')
REPO = WORK / 'mini-world-model'
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/anhsown/mini-world-model.git', str(REPO)], check=True)

BASE = REPO / 'research/mmad_model_benchmark'
DATA = WORK / 'mmad_locked140_data'
OUT = WORK / 'cosmos3_mmad_locked140_bnb8'
PRED_OUT = OUT / 'predictions.jsonl'
OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(BASE))
print('repo commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
import torch, transformers, PIL
from PIL import Image, ImageDraw, ImageFont  # verifies the modules used by torchvision/transformers

assert torch.cuda.is_available(), 'Enable GPU T4 x2 in Kaggle Session options.'
gpus = []
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    gpus.append({'index': i, 'name': p.name, 'vram_GiB': round(p.total_memory / 2**30, 2)})
print(json.dumps({'torch': torch.__version__, 'transformers': transformers.__version__, 'pillow': PIL.__version__, 'gpus': gpus}, indent=2))
assert torch.cuda.device_count() == 2, 'Select exactly GPU T4 x2 for this notebook.'
assert all('T4' in x['name'] for x in gpus), f'Expected two T4 GPUs, got {gpus}'


In [ ]:
# Build the frozen 20-per-task MMAD subset and materialize only its images.
subprocess.run([
    sys.executable, str(BASE / 'prepare_subset.py'),
    '--output', str(DATA),
    '--questions-per-task', '20',
    '--seed', '20260729',
    '--metadata-only',
], cwd=BASE, check=True)

from prepare_subset import materialize_images
manifest_path = DATA / 'subset_manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
records = manifest['records']
record_number = {row['sample_id']: i for i, row in enumerate(records, 1)}
assert len(records) == 140, len(records)
assert len({r['question_type'] for r in records}) == 7
assert all(sum(x['question_type'] == task for x in records) == 20
           for task in {r['question_type'] for r in records})
materialize_images(manifest, DATA / 'images')
print('locked manifest:', manifest['manifest_sha256'])
print('questions:', len(records), 'unique images:', len({r['image_file'] for r in records}))
print('tasks:', {t: sum(r['question_type'] == t for r in records)
                 for t in sorted({r['question_type'] for r in records})})


In [ ]:
# Optional Hugging Face authentication. Store HF_TOKEN in Kaggle Secrets; never paste it here.
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN')
if hf_token:
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
print('HF auth:', 'token enabled' if hf_token else 'anonymous')


# Optional GitHub write authentication for shared checkpoint shards.
# Create a Kaggle Secret named GITHUB_TOKEN; never paste the PAT into this notebook.
try:
    github_token = UserSecretsClient().get_secret('GITHUB_TOKEN')
except Exception:
    github_token = os.environ.get('GITHUB_TOKEN')
if github_token:
    os.environ['GITHUB_TOKEN'] = github_token
print('GitHub checkpoint push:', 'enabled' if github_token else 'read-only')


In [ ]:
# Load Cosmos 3 Nano Reasoner-only BNB8 across both T4 GPUs.
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

MODEL_ID = 'ThePyProgrammer/Cosmos3-Nano-reasoner-bnb8-vllm-und-only'
BASE_MODEL_ID = 'nvidia/Cosmos3-Nano'
MAX_NEW_TOKENS = 512  # enough room for UI-style reasoning; old notebook used only 32

free_gib = shutil.disk_usage(WORK).free / 2**30
print(f'free disk: {free_gib:.2f} GiB')
assert free_gib >= 11, 'Need at least 11 GiB free for the Reasoner-only checkpoint.'

processor = AutoProcessor.from_pretrained(
    MODEL_ID, min_pixels=256 * 28 * 28, max_pixels=512 * 28 * 28, token=hf_token
)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map='auto',
    max_memory={0: '14GiB', 1: '14GiB', 'cpu': '24GiB'},
    low_cpu_mem_usage=True,
    offload_folder=str(WORK / 'cosmos_offload'),
    offload_state_dict=True,
    attn_implementation='sdpa',
    token=hf_token,
).eval()

print('device map:', json.dumps(getattr(model, 'hf_device_map', {}), indent=2, default=str))
for i in range(2):
    print(f'cuda:{i} allocated GiB:', round(torch.cuda.memory_allocated(i) / 2**30, 2))


In [ ]:
from datetime import datetime, timezone
from common.mmad import (SYSTEM_PROMPT, append_jsonl, load_jsonl, parse_prediction,
                         evaluate_records, write_evaluation)
from PIL import Image
import re

def split_reasoning_response(text):
    cleaned = (text or '').replace('\r\n', '\n').strip()
    if not cleaned:
        return {'reasoning': '', 'response': '', 'parse_format': 'empty'}
    tagged = re.search(r'<think>\s*(.*?)\s*</think>\s*(.*)', cleaned, re.I | re.S)
    if tagged:
        return {'reasoning': tagged.group(1).strip(),
                'response': tagged.group(2).strip(), 'parse_format': 'think_tags'}
    marker = re.search(r'\nResponse\s*\n', cleaned, re.I)
    if marker:
        return {'reasoning': cleaned[:marker.start()].strip(),
                'response': cleaned[marker.end():].strip(), 'parse_format': 'nvidia_ui'}
    prediction = parse_prediction(cleaned)
    return {'reasoning': '', 'response': prediction or cleaned,
            'parse_format': 'answer_only' if prediction else 'unstructured'}

input_device = model.device

def materialize_batch(selected):
    batch_manifest = {'records': selected}
    materialize_all_images(batch_manifest, DATA, CACHE, range_download=True)
    missing = [r['image_file'] for r in selected if not (DATA / r['image_file']).exists()]
    assert not missing, f'{len(missing)} batch images are missing'

def cleanup_batch(selected):
    for relative in {r['image_file'] for r in selected}:
        (DATA / relative).unlink(missing_ok=True)

def group_by_image_batches(selected, images_per_batch=64):
    groups = {}
    order = []
    for row in selected:
        key = row['image_file']
        if key not in groups:
            groups[key] = []
            order.append(key)
        groups[key].append(row)
    for start in range(0, len(order), images_per_batch):
        keys = order[start:start + images_per_batch]
        yield [row for key in keys for row in groups[key]]

def sync_cuda():
    for i in range(torch.cuda.device_count()):
        torch.cuda.synchronize(i)

def ui_style_output(reasoning, response):
    reasoning = (reasoning or '').strip()
    response = (response or '').strip()
    return f'<think>\n{reasoning}\n</think>\n\n{response}'.strip()

def infer_one(sample):
    image_path = (DATA / sample['image_file']).resolve()
    conversation = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': [
            {'type': 'image', 'image': str(image_path)},
            {'type': 'text', 'text': sample['prompt']},
        ]},
    ]
    chat = processor.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True)
    with Image.open(image_path) as source:
        image_input = source.convert('RGB')
        inputs = processor(text=[chat], images=[image_input],
                           padding=True, return_tensors='pt').to(input_device)
    seed = 20260731 + int(sample['sample_id'].rsplit('_', 1)[-1])
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    sync_cuda(); started = time.perf_counter()
    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
        )
    sync_cuda(); latency = time.perf_counter() - started
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated)]
    raw = processor.batch_decode(trimmed, skip_special_tokens=True,
                                 clean_up_tokenization_spaces=False)[0].strip()
    parts = split_reasoning_response(raw)
    prediction = parse_prediction(parts['response']) or parse_prediction(raw)
    normalized = ui_style_output(parts['reasoning'], parts['response'])
    return raw, normalized, parts, prediction, latency

def run_range(selected, output_path, label, deadline=None):
    previous = load_jsonl(output_path)
    done = {r['sample_id'] for r in previous if r.get('status') == 'ok'}
    pending = [r for r in selected if r['sample_id'] not in done]
    print(f'{label}: total={len(selected)} completed={len(selected)-len(pending)} pending={len(pending)}', flush=True)
    all_started = time.perf_counter()
    for position, sample in enumerate(pending, 1):
        if deadline is not None and time.perf_counter() >= deadline:
            print(f'TIME BUDGET REACHED before question {record_number[sample["sample_id"]]}', flush=True)
            break
        try:
            raw, normalized, parts, pred, latency = infer_one(sample)
            status, error = ('ok' if pred else 'parse_failure'), None
        except Exception as exc:
            if isinstance(exc, torch.OutOfMemoryError):
                torch.cuda.empty_cache()
            raw = normalized = ''
            parts = {'reasoning': '', 'response': '', 'parse_format': 'error'}
            pred, latency, status, error = None, 0.0, 'error', f'{type(exc).__name__}: {exc}'
        row = {
            'sample_id': sample['sample_id'],
            'question_number': record_number[sample['sample_id']],
            'model': MODEL_ID,
            'base_model': BASE_MODEL_ID,
            'precision': 'community BNB8 reasoner-only',
            'backend': 'Transformers/device_map=auto/T4x2',
            'manifest_sha256': manifest['manifest_sha256'],
            'benchmark_split': 'locked140_seed20260729',
            'question_type': sample['question_type'],
            'source_dataset': sample['source_dataset'],
            'category': sample['category'],
            'status': status,
            'prediction': pred,
            'raw_response': raw,
            'ui_style_output': normalized,
            'reasoning': parts['reasoning'],
            'response': parts['response'],
            'parse_format': parts['parse_format'],
            'latency_seconds': round(latency, 4),
            'error': error,
            'created_at': datetime.now(timezone.utc).isoformat(),
        }
        append_jsonl(output_path, row)
        truth = sample['answer']
        print(f'[{row["question_number"]}/{len(records)}] {status.upper()} pred={pred} truth={truth} '
              f'correct={pred == truth} {latency:.1f}s', flush=True)
        if 'SMOKE' in label:
            print('--- UI-STYLE OUTPUT ---')
            print(normalized or f'ERROR: {error}')
            print('--- PARSED ---')
            print(json.dumps({'reasoning': parts['reasoning'], 'response': parts['response'],
                              'prediction': pred, 'truth': truth}, ensure_ascii=False, indent=2))
        if position % 25 == 0:
            elapsed = time.perf_counter() - all_started
            eta_h = ((len(pending)-position) * elapsed / max(position, 1)) / 3600
            print(f'checkpoint={output_path} ETA={eta_h:.2f}h', flush=True)
    return load_jsonl(output_path)


In [ ]:
# Smoke gate: run the first five locked questions and inspect complete reasoning.
smoke_records = records[:5]
smoke_predictions = run_range(smoke_records, PRED_OUT, 'LOCKED140 SMOKE 1-5')
latest = {r['sample_id']: r for r in smoke_predictions}
smoke_latest = [latest[r['sample_id']] for r in smoke_records if r['sample_id'] in latest]
coverage = sum(r.get('status') == 'ok' for r in smoke_latest) / 5
reasoning_rate = sum(bool((r.get('reasoning') or '').strip()) for r in smoke_latest) / 5
SMOKE_GATE = {
    'attempted': len(smoke_latest),
    'parseable_output_coverage': coverage,
    'nonempty_reasoning_rate': reasoning_rate,
    'valid': len(smoke_latest) == 5 and coverage >= 0.8 and reasoning_rate >= 0.8,
}
print(json.dumps(SMOKE_GATE, indent=2))
(OUT / 'smoke_gate.json').write_text(json.dumps(SMOKE_GATE, indent=2), encoding='utf-8')


## Locked benchmark gate

Continue only when image/question pairing is correct and at least four of the five smoke answers are parseable. The benchmark cell resumes automatically from `predictions.jsonl`.


In [ ]:
# Run all 140 locked questions. Existing successful smoke rows are skipped.
assert SMOKE_GATE['valid'], f'Smoke gate failed: {SMOKE_GATE}'
MAX_RUNTIME_HOURS = 4.0
run_started = time.perf_counter()
deadline = run_started + MAX_RUNTIME_HOURS * 3600

previous = load_jsonl(PRED_OUT)
completed_ids = {r['sample_id'] for r in previous if r.get('status') == 'ok'}
remaining = [r for r in records if r['sample_id'] not in completed_ids]
batches = list(group_by_image_batches(remaining, images_per_batch=16))
print(f'locked=140 resumed_ok={len(completed_ids)} remaining={len(remaining)} batches={len(batches)}')

for batch_index, batch in enumerate(batches, 1):
    if time.perf_counter() >= deadline:
        print('STOPPED: four-hour safety budget reached; rerun this cell to resume.', flush=True)
        break
    print(f'\n=== LOCKED BATCH {batch_index}/{len(batches)}: {len(batch)} questions, '
          f'{len({r["image_file"] for r in batch})} images ===', flush=True)
    run_range(batch, PRED_OUT, f'LOCKED140 batch {batch_index}', deadline=deadline)

predictions = load_jsonl(PRED_OUT)
summary, scored = evaluate_records(manifest, predictions)
summary['model_label'] = 'Cosmos 3 Nano Reasoner community BNB8'
summary['benchmark_split'] = 'locked140_seed20260729'
summary['decoding'] = 'greedy_do_sample_false'
summary['runtime_hours_this_session'] = round((time.perf_counter() - run_started) / 3600, 4)
write_evaluation(OUT, summary, scored)
print(json.dumps(summary, ensure_ascii=False, indent=2))
archive = shutil.make_archive('/kaggle/working/cosmos3_mmad_locked140_bnb8_results', 'zip', OUT)
print('DOWNLOAD:', archive)


In [ ]:
# Rebuild a portable archive at any time after rerunning the benchmark cell.
archive = shutil.make_archive('/kaggle/working/cosmos3_mmad_locked140_bnb8_results', 'zip', OUT)
print('DOWNLOAD:', archive)
print('size MiB:', round(Path(archive).stat().st_size / 2**20, 2))
print('checkpoint:', PRED_OUT)
